# 02 — Data Cleaning
This notebook handles missing values, invalid values, exact duplicates, cancellations, and review flags using explicit rules. The original Excel file is never modified.

## Learning objectives
- Preserve identifiers and missing values correctly
- Understand numeric/date parsing and the duplicate decision
- Separate eligible sales, cancellations, and flagged rows

In [1]:
from retail_analysis.data_loader import load_retail_data
from retail_analysis.cleaning import clean_transactions, split_transaction_sets
from retail_analysis.validation import data_quality_report, issue_counts

raw = load_retail_data()
cleaned, duplicate_stats = clean_transactions(raw)
sales, cancellations, flagged = split_transaction_sets(cleaned)
duplicate_stats

{'input_rows': 541909,
 'duplicate_candidates': 10147,
 'duplicate_rows_removed': 5268,
 'output_rows': 536641}

`pd.to_numeric(..., errors='coerce')` and `pd.to_datetime(..., errors='coerce')` turn values that cannot be parsed into missing values. The code also flags them, so the problems are not hidden. The Boolean setting in `config.py` controls exact-duplicate removal.

In [2]:
data_quality_report(cleaned, 'After cleaning')

,Stage,Column,DataType,RowCount,MissingCount,MissingPercent
0,After cleaning,InvoiceNo,string,536641,0,0.00
1,After cleaning,StockCode,string,536641,0,0.00
2,After cleaning,Description,string,536641,421,0.08
3,After cleaning,Quantity,int64,536641,0,0.00
4,After cleaning,InvoiceDate,datetime64[ns],536641,0,0.00
5,After cleaning,UnitPrice,float64,536641,0,0.00
6,After cleaning,CustomerID,string,536641,135037,25.16
7,After cleaning,Country,string,536641,0,0.00
8,After cleaning,InvalidQuantity,bool,536641,0,0.00
9,After cleaning,InvalidUnitPrice,bool,536641,0,0.00


In [3]:
issue_counts(cleaned)

,Issue,Count
0,ExactDuplicateCandidates,0
1,InvalidDate,0
2,InvalidQuantity,0
3,InvalidUnitPrice,0
4,ZeroQuantity,0
5,NegativeQuantity,10587
6,ZeroOrNegativePrice,2512
7,IsCancellation,9251
8,MissingDescription,421
9,NeedsReview,11763


Anonymous eligible sales are retained and excluded only from customer-level analysis. An invoice with a `C` prefix is a cancellation. Negative quantities are flagged separately instead of assuming that every one is a cancellation.

In [4]:
retention = {
    'cleaned': len(cleaned), 'eligible_sales': len(sales),
    'cancellations': len(cancellations), 'flagged_for_review': len(flagged)
}
retention

{'cleaned': 536641,
 'eligible_sales': 524878,
 'cancellations': 9251,
 'flagged_for_review': 11763}

## Interpretation and practice questions
Flagged rows are not deleted automatically, and the outputs may overlap. A missing description is filled only when a StockCode has exactly one known description.

1. What is the most common reason for `NeedsReview`?
2. How many negative-quantity rows do not have a C prefix?
3. How does the row count change when duplicate removal is set to False?